[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oshbocker/orbit_wars/blob/main/notebooks/transformer_dagger_walkthrough.ipynb)

# The Transformer DAgger PPO Agent: A Complete Walkthrough

**An Andrej Karpathy-style pedagogical guide to understanding every layer of our RL agent.**

---

Most RL tutorials show you the algorithm and wave their hands at the architecture. This notebook does the opposite: we're going to **run real code, print real tensors, and watch data flow through every single layer** of our Transformer DAgger PPO agent for the Orbit Wars competition.

By the end, you'll understand:
1. How raw game observations become structured features
2. How the transformer processes those features into action distributions
3. How PPO's clipped objective trains the policy
4. How DAgger bootstraps learning from an expert
5. Where this architecture is strong, where it's weak, and what to try next

> **Philosophy**: The best way to understand a neural network is to watch numbers flow through it. Abstract diagrams lie; tensors don't.

## Part 0: Setup

In [ ]:
#@title Install Dependencies & Clone Repo (Colab only)
#@markdown Run this cell if you're on Google Colab. Skip if running locally.

import os, sys

IN_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ

if IN_COLAB:
    REPO_URL = 'https://github.com/oshbocker/orbit_wars.git'
    REPO_DIR = '/content/orbit_wars'

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    else:
        print(f'Repo already cloned at {REPO_DIR}')

    os.chdir(REPO_DIR)
    sys.path.insert(0, REPO_DIR)

    !pip install -q --upgrade "kaggle-environments>=1.28.0"
    !pip install -q gymnasium pyyaml

    print(f'Working dir: {os.getcwd()}')
    print('Colab setup complete.')
else:
    print('Not in Colab — skipping. Make sure you run from the repo root.')


In [ ]:
import sys, os
# Ensure repo root is on path (handles both local and Colab)
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
from torch.distributions import Categorical

# Our modules
from src.config import load_train_config, EnvConfig, ModelConfig
from src.game_types import parse_observation, PlanetState, FleetState, GameState
from src.features import (
    encode_source_decision, build_global_features, compute_fleet_transit,
    fleet_speed, passes_through_sun, safe_angle, planet_pos_at,
    GLOBAL_DIM, SOURCE_SCALAR_DIM, KNN_SCALAR_DIM, TARGET_SCALAR_DIM,
    SourceDecision, FleetTransitState
)
from src.policy import TransformerPolicy
from src.ppo import sample_actions, action_log_prob_and_entropy

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cpu')  # CPU for inspection

# Load the DAgger config
cfg = load_train_config('configs/transformer_dagger.yaml')
print(f"Config loaded: {cfg.run_name}")
print(f"Model: embed_dim={cfg.model.embed_dim}, n_layers={cfg.model.n_layers}, n_heads={cfg.model.n_heads}")
print(f"Env: max_targets={cfg.env.max_targets}, k_neighbors={cfg.env.k_neighbors}")
print(f"PPO: lr={cfg.ppo.lr}, clip={cfg.ppo.clip_coef}, epochs={cfg.ppo.epochs}")
print(f"Imitation: bc_games={cfg.imitation.bc_games}, coef_start={cfg.imitation.coef_start}")


---

## Part 1: The Game State — What the Agent Sees

Before we touch any neural network, we need to understand **what the agent actually observes**. In Orbit Wars, you control fleets on a 2D board with a deadly sun at the center. Let's create a realistic game state and see exactly what the raw observation looks like.

> **Key insight**: Feature engineering is 80% of RL performance. The network can only learn patterns that exist in its inputs. If you don't encode fleet transit times, the network can never learn to time attacks.

In [ ]:
# Let's create a realistic game state manually so we can inspect everything
# In a real game, this comes from the Kaggle environment

# Simulate a mid-game scenario: we own 3 planets, enemy owns 2, 2 neutral
mock_planets = [
    # [id, owner, x, y, radius, ships, production]
    [0, 0, 25.0, 25.0, 2.6, 45, 5],   # Ours - home planet, high production
    [1, 0, 30.0, 70.0, 2.1, 20, 3],   # Ours - forward base
    [2, 0, 15.0, 40.0, 1.7, 12, 2],   # Ours - small outpost
    [3, 1, 75.0, 75.0, 2.6, 50, 5],   # Enemy - home planet
    [4, 1, 70.0, 30.0, 2.1, 30, 3],   # Enemy - forward base
    [5, -1, 50.0, 20.0, 1.0, 8, 1],   # Neutral - near sun!
    [6, -1, 60.0, 60.0, 1.7, 15, 2],  # Neutral - contested
]

mock_fleets = [
    # [id, owner, x, y, angle, from_planet_id, ships]
    [0, 0, 35.0, 50.0, 0.78, 0, 10],  # Our fleet heading NE
    [1, 1, 65.0, 50.0, 3.14, 3, 15],  # Enemy fleet heading W
]

# Build a mock observation (as the Kaggle env would provide)
class MockObs:
    def __init__(self):
        self.player = 0
        self.step = 120  # mid-game
        self.planets = mock_planets
        self.fleets = mock_fleets
        self.angular_velocity = 0.03
        self.initial_planets = mock_planets  # simplified
        self.comets = None
        self.comet_planet_ids = []

obs = MockObs()
state = parse_observation(obs)

print("=== Parsed Game State ===")
print(f"Step: {state.step}/500  |  We are Player {state.player}")
print(f"Angular velocity: {state.angular_velocity} rad/turn")
print(f"\nPlanets ({len(state.planets)}):")
for p in state.planets:
    owner_str = {-1: 'NEUTRAL', 0: 'OURS', 1: 'ENEMY'}[p.owner]
    orbit_str = ' [ORBITING]' if p.is_orbiting else ''
    print(f"  Planet {p.id}: {owner_str:>7} at ({p.x:.0f},{p.y:.0f}) "
          f"ships={p.ships} prod={p.production}{orbit_str}")

print(f"\nFleets ({len(state.fleets)}):")
for f in state.fleets:
    owner_str = 'OURS' if f.owner == 0 else 'ENEMY'
    print(f"  Fleet {f.id}: {owner_str} at ({f.x:.0f},{f.y:.0f}) "
          f"ships={f.ships} angle={f.angle:.2f}rad")

### 1.1 Fleet Physics: Speed, Sun Avoidance, and Transit

One of the most important things to understand: **fleet speed is logarithmic in ship count**. Sending 1 ship is painfully slow (1 unit/turn). Sending 100 ships is ~3x faster. This creates a deep strategic tension: split your forces for flexibility, or concentrate them for speed?

> **Wisdom**: This speed formula is the single most important game mechanic to internalize. It means that a 500-ship fleet arrives in roughly the same time as a 50-ship fleet would cover half the distance. Concentration of force is rewarded.

In [ ]:
# Fleet speed as a function of ship count
ship_counts = [1, 5, 10, 25, 50, 100, 200, 500, 1000]
print("Ships -> Speed (units/turn) -> Time to cross 50 units")
print("-" * 55)
for s in ship_counts:
    spd = fleet_speed(s)
    time = 50.0 / spd
    bar = '#' * int(spd * 5)
    print(f"  {s:>5} ships -> {spd:.2f} u/t -> {time:.1f} turns  {bar}")

print("\n--- Sun Avoidance ---")
# Can we send a fleet from (25,25) to (75,75)? That goes right through the sun!
src_x, src_y = 25.0, 25.0
dst_x, dst_y = 75.0, 75.0
blocked = passes_through_sun(src_x, src_y, dst_x, dst_y)
print(f"Direct path (25,25)->(75,75): {'BLOCKED by sun!' if blocked else 'Clear'}")

# The safe_angle function finds a waypoint around the sun
angle, was_redirected = safe_angle(src_x, src_y, dst_x, dst_y)
print(f"Safe angle: {angle:.3f} rad ({math.degrees(angle):.1f} deg)")
print(f"Was redirected around sun: {was_redirected}")

# A path that doesn't cross the sun
blocked2 = passes_through_sun(25.0, 25.0, 30.0, 70.0)
print(f"\nDirect path (25,25)->(30,70): {'BLOCKED' if blocked2 else 'Clear - no sun issue'}")

### 1.2 Fleet Transit: Who's Going Where?

Before the agent decides where to send ships, it needs to know **what's already in flight**. For every fleet on the board, we use ray-circle intersection to predict which planet it will hit and when.

This is aggregated per-planet as `(total_enemy_ships, avg_enemy_eta, total_friendly_ships, avg_friendly_eta)`. The agent uses this to avoid sending reinforcements to planets that are already well-supplied, or to urgently defend planets under attack.

> **Key insight**: Transit features are the agent's "memory" of what it has already committed. Without them, the agent would keep sending fleets to the same target every turn.

In [ ]:
transit = compute_fleet_transit(state)

print("=== Fleet Transit State ===")
print("(For each planet: incoming ships and estimated arrival time)\n")

for pid, info in sorted(transit.transit.items()):
    planet = state.planets_by_id[pid]
    owner_str = {-1: 'NEU', 0: 'OUR', 1: 'ENE'}[planet.owner]
    parts = []
    if info.enemy_ships > 0:
        parts.append(f"enemy: {info.enemy_ships:.0f} ships in {info.enemy_eta:.1f}t")
    if info.friendly_ships > 0:
        parts.append(f"friendly: {info.friendly_ships:.0f} ships in {info.friendly_eta:.1f}t")
    if parts:
        print(f"  Planet {pid} [{owner_str}]: {' | '.join(parts)}")

if not transit.transit:
    print("  (No fleets predicted to hit any planet in this simple example)")
    print("  In real games, this dict is usually populated with 5-15 entries.")

# Show how transit state gets updated sequentially
print("\n--- Sequential Transit Updates ---")
print("When planet A decides to send 20 ships to planet 6 (neutral):")
transit.add_fleet(6, 20.0, 8.5, is_friendly=True)
info = transit.transit[6]
print(f"  Planet 6 transit: friendly={info.friendly_ships:.0f} ships, eta={info.friendly_eta:.1f}t")

print("When planet B ALSO sends 10 ships to planet 6:")
transit.add_fleet(6, 10.0, 5.0, is_friendly=True)
info = transit.transit[6]
print(f"  Planet 6 transit: friendly={info.friendly_ships:.0f} ships, eta={info.friendly_eta:.1f}t")
print(f"  (ETA is weighted average: (20*8.5 + 10*5.0) / 30 = {(20*8.5+10*5.0)/30:.1f}t)")

---

## Part 2: Feature Encoding — From Game State to Tensors

This is where domain knowledge meets neural networks. We need to convert the rich, variable-size game state into fixed-size tensors that a transformer can process.

The architecture makes a **critical design choice**: instead of encoding the entire board at once, we make **one decision per owned planet**, processing planets from most ships to fewest. Each decision sees:

| Input | Shape | What it captures |
|-------|-------|------------------|
| Global features | `[9]` | Game phase, relative strength |
| Source planet | `[2] + [7]` | Position + properties of the planet we're deciding for |
| K nearest neighbors | `[3, 2] + [3, 4]` | Local neighborhood context |
| Target planets | `[30, 2] + [30, 11]` | All candidate destinations |
| Target mask | `[32]` | Which targets are valid (not blocked by sun) |

> **Why per-planet?** This is an inductive bias. We're telling the network: "each planet makes its own shipping decision." This dramatically reduces the action space compared to outputting all moves at once.

In [ ]:
# Reset transit for clean demo
transit = compute_fleet_transit(state)

# Get our planets sorted by ship count (descending) — this is the decision order
my_planets = sorted(
    [p for p in state.planets if p.owner == state.player],
    key=lambda p: -p.ships
)
print("=== Decision Order (most ships first) ===")
for i, p in enumerate(my_planets):
    print(f"  Decision {i}: Planet {p.id} ({p.ships} ships, prod={p.production})")

# Encode the FIRST decision (our strongest planet)
src = my_planets[0]
decision = encode_source_decision(src, state, transit, cfg.env)

print(f"\n=== SourceDecision for Planet {src.id} ===")
print(f"Source: Planet {decision.source_id} with {decision.remaining_ships} ships")
print(f"Number of valid targets: {sum(decision.target_mask) - 2} (+ CLS + NoOp)")
print(f"Target planet IDs: {decision.target_planet_ids}")

### 2.1 Global Features — The Big Picture in 9 Numbers

The global features capture **where we stand in the game**. Are we winning? Losing? Early game or late game?

In [ ]:
g = decision.global_features
labels = [
    'step / 500 (game progress)',
    'my_ships (normalized)',
    'enemy1_ships (normalized)',
    'enemy2_ships (normalized)',
    'enemy3_ships (normalized)',
    'my_production (normalized)',
    'enemy1_production (normalized)',
    'enemy2_production (normalized)',
    'enemy3_production (normalized)',
]

print(f"Global Features [{GLOBAL_DIM}]:")
print("=" * 60)
for i, (val, label) in enumerate(zip(g, labels)):
    bar = '|' + '#' * int(val * 40) + ' ' * (40 - int(val * 40)) + '|'
    print(f"  [{i}] {val:.4f}  {bar}  {label}")

print(f"\nInterpretation:")
print(f"  - Game is {g[0]*100:.0f}% complete (step {int(g[0]*500)}/500)")
print(f"  - We have {g[1]:.1%} of max ship count")
print(f"  - Our production is {g[5]:.1%} of max")

### 2.2 Source Planet Features — Where Are We Sending From?

For the planet making the decision, we encode its position separately from its scalar properties. **Why?** Because positions get a special shared encoder (a 2-layer MLP) that learns a spatial embedding. This shared encoder is used for source, KNN, and target positions — so the network develops a unified "sense of space."

> **Design choice**: Separating position from other scalars and sharing the position encoder is an inductive bias that says "spatial relationships matter the same way regardless of which planet type we're encoding."

In [ ]:
print("Source Position [2]:")
print(f"  x/100 = {decision.source_position[0]:.4f}  (raw: {decision.source_position[0]*100:.1f})")
print(f"  y/100 = {decision.source_position[1]:.4f}  (raw: {decision.source_position[1]*100:.1f})")

s = decision.source_scalars
src_labels = [
    ('radius / 5.0', 5.0),
    ('production / 5.0', 5.0),
    ('log1p(ships) / 10.0', None),
    ('log1p(enemy_transit) / 10.0', None),
    ('enemy_eta / 50.0', 50.0),
    ('log1p(friendly_transit) / 10.0', None),
    ('friendly_eta / 50.0', 50.0),
]

print(f"\nSource Scalars [{SOURCE_SCALAR_DIM}]:")
print("=" * 60)
for i, (val, (label, scale)) in enumerate(zip(s, src_labels)):
    raw = f" (raw: {val * scale:.1f})" if scale else ""
    if 'log1p' in label and val > 0:
        raw = f" (raw ships: {math.expm1(val * 10):.0f})"
    print(f"  [{i}] {val:.4f}  {label}{raw}")

print(f"\nInterpretation: Planet {src.id} has {src.ships} ships, produces {src.production}/turn")
print(f"Transit features are 0 because no fleets are heading to this planet (yet).")

### 2.3 KNN Features — What's In the Neighborhood?

The K=3 nearest neighbors give the network **local context**. Is our planet in a cluster of friendly planets (safe to send ships away) or isolated near enemies (need to keep a garrison)?

These get mean-pooled into a single vector, so the network sees the **average neighborhood** rather than individual neighbors.

In [ ]:
K = cfg.env.k_neighbors
print(f"KNN: {K} nearest neighbors to Planet {src.id}")
print("=" * 60)

knn_labels = ['radius/5', 'prod/5', 'log1p(ships)/10', 'is_orbiting']

for k in range(K):
    pos = decision.knn_positions[k]
    scalars = decision.knn_scalars[k]
    dist = math.hypot(
        (pos[0] - decision.source_position[0]) * 100,
        (pos[1] - decision.source_position[1]) * 100
    )
    print(f"\n  Neighbor {k} at ({pos[0]*100:.0f}, {pos[1]*100:.0f}) — distance: {dist:.1f} units")
    for i, (val, label) in enumerate(zip(scalars, knn_labels)):
        print(f"    [{i}] {val:.4f}  {label}")

print(f"\nMean-pooled KNN scalars: {decision.knn_scalars.mean(axis=0)}")
print("(This mean-pooled vector is what the network actually sees after encoding)")

### 2.4 Target Features — Every Possible Destination

The T=30 closest planets become candidate targets. Each target gets 11 scalar features plus its position. The network will attend over these targets using a transformer and produce a probability distribution: "which planet should I send ships to?"

> **Design choice**: 30 targets is plenty — in most games there are 20-40 planets total, and many are too far to matter. The sorting by distance is another inductive bias: closer targets are more likely to be relevant.

In [ ]:
T = cfg.env.max_targets
n_valid = len(decision.target_planet_ids)
print(f"Targets: {n_valid} valid out of {T} slots")
print("=" * 80)

tgt_labels = [
    'is_neutral', 'is_friendly', 'is_enemy', 'dist/100',
    'log1p(ships)/10', 'prod/5', 'log1p(enemy_transit)/10',
    'enemy_eta/50', 'log1p(friendly_transit)/10', 'friendly_eta/50',
    'is_orbiting'
]

for t in range(min(n_valid, 6)):  # Show first 6 targets
    pid = decision.target_planet_ids[t]
    planet = state.planets_by_id[pid]
    pos = decision.target_positions[t]
    scalars = decision.target_scalars[t]
    owner_str = {-1: 'NEUTRAL', 0: 'OURS', 1: 'ENEMY'}[planet.owner]
    
    print(f"\n  Target {t} = Planet {pid} [{owner_str}] "
          f"at ({pos[0]*100:.0f},{pos[1]*100:.0f})")
    for i, (val, label) in enumerate(zip(scalars, tgt_labels)):
        if val != 0:
            print(f"    [{i:>2}] {val:.4f}  {label}")

print(f"\n  ... ({n_valid - 6} more targets)" if n_valid > 6 else "")

print(f"\nTarget mask (first 10): {decision.target_mask[:10]}")
print(f"  [0]=CLS (always True), [1]=NoOp (always True), [2:]=targets")

### 2.5 Complete Feature Summary

Let's see all the tensor shapes that will feed into the neural network:

In [ ]:
print("=== Complete Feature Shapes (one SourceDecision) ===")
print(f"  global_features:    {decision.global_features.shape}   = {GLOBAL_DIM} game-level numbers")
print(f"  source_position:    {decision.source_position.shape}       = (x, y) normalized")
print(f"  source_scalars:     {decision.source_scalars.shape}       = {SOURCE_SCALAR_DIM} planet properties")
print(f"  knn_positions:      {decision.knn_positions.shape}     = {K} neighbors x 2 coords")
print(f"  knn_scalars:        {decision.knn_scalars.shape}     = {K} neighbors x {KNN_SCALAR_DIM} scalars")
print(f"  target_positions:   {decision.target_positions.shape}   = {T} targets x 2 coords")
print(f"  target_scalars:     {decision.target_scalars.shape}  = {T} targets x {TARGET_SCALAR_DIM} scalars")
print(f"  target_mask:        {decision.target_mask.shape}    = CLS + NoOp + {T} targets")
print(f"")
total_floats = (GLOBAL_DIM + 2 + SOURCE_SCALAR_DIM + K*2 + K*KNN_SCALAR_DIM 
                + T*2 + T*TARGET_SCALAR_DIM + T+2)
print(f"  Total input numbers per decision: {total_floats}")
print(f"  For comparison, an image classifier sees 224*224*3 = {224*224*3:,} pixels")
print(f"  Our input is tiny but densely packed with domain knowledge!")

---

## Part 3: The Transformer Policy — From Features to Actions

Now for the main event. Let's instantiate the policy network and watch data flow through every layer.

The architecture has 5 stages:

```
Raw Features
    -> Encoders (5 MLPs, one shared position encoder)
        -> Token Construction (CLS + NoOp + Targets)
            -> Transformer (2 layers, 4 heads)
                -> Output Heads (target logits, fraction logits, value)
```

> **Architecture philosophy**: This is NOT a standard vision transformer or NLP transformer. It's a **decision transformer** where each token represents a possible action (target planet), and the CLS token provides a state value estimate. The transformer's self-attention lets targets "communicate" — e.g., if one target is very attractive, it suppresses others.

In [ ]:
# Create the policy
policy = TransformerPolicy(cfg.model, cfg.env).to(device)
policy.eval()  # no dropout

# Count parameters by component
def count_params(module):
    return sum(p.numel() for p in module.parameters())

print("=== TransformerPolicy Architecture ===")
print(f"Total parameters: {count_params(policy):,}\n")

components = [
    ('pos_encoder (shared)', policy.pos_encoder),
    ('global_encoder', policy.global_encoder),
    ('source_encoder', policy.source_encoder),
    ('knn_encoder', policy.knn_encoder),
    ('source_knn_combiner', policy.source_knn_combiner),
    ('target_encoder', policy.target_encoder),
    ('token_projection', policy.token_projection),
    ('transformer_blocks', policy.transformer_blocks),
    ('final_ln', policy.final_ln),
    ('target_head', policy.target_head),
    ('fraction_head', policy.fraction_head),
    ('value_head', policy.value_head),
]

for name, module in components:
    params = count_params(module)
    pct = params / count_params(policy) * 100
    bar = '#' * int(pct / 2)
    print(f"  {name:<25} {params:>7,} params ({pct:>5.1f}%)  {bar}")


### 3.1 Stage 1: The Encoders — Projecting to Embedding Space

Each input type gets its own MLP encoder that projects it into the shared `embed_dim=128` space. Let's trace through each one.

**The Position Encoder** is particularly elegant: it's a shared 2-layer MLP `(2 -> 64 -> 128)` that converts (x, y) coordinates into a learned spatial embedding. The same encoder is used for source, KNN, and target positions.

> **Why share the position encoder?** Transfer learning within the model. A planet at (30, 70) should have the same spatial embedding whether it's a source, neighbor, or target. Sharing enforces this.

In [ ]:
# Prepare tensors (batch size 1)
B = 1
global_t = torch.tensor(decision.global_features, dtype=torch.float32).unsqueeze(0)
src_scalars_t = torch.tensor(decision.source_scalars, dtype=torch.float32).unsqueeze(0)
src_pos_t = torch.tensor(decision.source_position, dtype=torch.float32).unsqueeze(0)
knn_scalars_t = torch.tensor(decision.knn_scalars, dtype=torch.float32).unsqueeze(0)
knn_pos_t = torch.tensor(decision.knn_positions, dtype=torch.float32).unsqueeze(0)
tgt_scalars_t = torch.tensor(decision.target_scalars, dtype=torch.float32).unsqueeze(0)
tgt_pos_t = torch.tensor(decision.target_positions, dtype=torch.float32).unsqueeze(0)
mask_t = torch.tensor(decision.target_mask, dtype=torch.bool).unsqueeze(0)

print("=== Stage 1: Encoding ===")

with torch.no_grad():
    # Position encoding (shared)
    src_pos_emb = policy.pos_encoder(src_pos_t)  # [1, 128]
    print(f"\n1a. Position Encoder: {list(src_pos_t.shape)} -> {list(src_pos_emb.shape)}")
    print(f"    Input (x,y):  {src_pos_t[0].numpy()}")
    print(f"    Output (first 8 dims): {src_pos_emb[0, :8].numpy().round(3)}")
    print(f"    Output range: [{src_pos_emb.min():.3f}, {src_pos_emb.max():.3f}]")
    
    # Global encoding
    global_emb = policy.global_encoder(global_t)  # [1, 128]
    print(f"\n1b. Global Encoder: {list(global_t.shape)} -> {list(global_emb.shape)}")
    print(f"    Output range: [{global_emb.min():.3f}, {global_emb.max():.3f}]")
    
    # Source encoding: concat(pos_emb, scalars) -> source_encoder
    src_input = torch.cat([src_pos_emb, src_scalars_t], dim=-1)  # [1, 128+7=135]
    src_emb = policy.source_encoder(src_input)  # [1, 128]
    print(f"\n1c. Source Encoder: concat(pos_emb[128], scalars[7]) = [{src_input.shape[-1]}] -> {list(src_emb.shape)}")
    print(f"    Output range: [{src_emb.min():.3f}, {src_emb.max():.3f}]")

In [ ]:
with torch.no_grad():
    # KNN encoding + mean pooling
    K = cfg.env.k_neighbors
    knn_pos_flat = knn_pos_t.reshape(B * K, 2)
    knn_pos_emb = policy.pos_encoder(knn_pos_flat).reshape(B, K, -1)  # [1, 3, 128]
    knn_input = torch.cat([knn_pos_emb, knn_scalars_t], dim=-1)  # [1, 3, 132]
    knn_emb = policy.knn_encoder(knn_input)  # [1, 3, 128]
    knn_pool = knn_emb.mean(dim=1)  # [1, 128] <-- mean over 3 neighbors
    
    print(f"1d. KNN Encoder:")
    print(f"    Per-neighbor: concat(pos_emb[128], scalars[4]) = [132] -> [128]")
    print(f"    Before pooling: {list(knn_emb.shape)} (3 neighbor embeddings)")
    print(f"    After mean pool: {list(knn_pool.shape)} (single neighborhood vector)")
    print(f"    Output range: [{knn_pool.min():.3f}, {knn_pool.max():.3f}]")
    
    # Combine source + KNN
    src_knn_input = torch.cat([src_emb, knn_pool], dim=-1)  # [1, 256]
    src_knn = policy.source_knn_combiner(src_knn_input)  # [1, 128]
    print(f"\n1e. Source+KNN Combiner: concat([128], [128]) = [256] -> {list(src_knn.shape)}")
    
    # Target encoding
    T = cfg.env.max_targets
    tgt_pos_flat = tgt_pos_t.reshape(B * T, 2)
    tgt_pos_emb = policy.pos_encoder(tgt_pos_flat).reshape(B, T, -1)  # [1, 30, 128]
    tgt_input = torch.cat([tgt_pos_emb, tgt_scalars_t], dim=-1)  # [1, 30, 139]
    tgt_emb = policy.target_encoder(tgt_input)  # [1, 30, 128]
    
    print(f"\n1f. Target Encoder:")
    print(f"    Per-target: concat(pos_emb[128], scalars[11]) = [139] -> [128]")
    print(f"    All targets: {list(tgt_emb.shape)}")
    print(f"    Output range: [{tgt_emb.min():.3f}, {tgt_emb.max():.3f}]")

### 3.2 Stage 2: Token Construction — Building the Sequence

Now comes the key architectural idea. We construct a **sequence of tokens** for the transformer:

```
[CLS] [NoOp] [Target_0] [Target_1] ... [Target_29] [PAD...]
```

- **CLS**: A learnable parameter (like BERT's [CLS]). Used only for the value head.
- **NoOp**: A learnable parameter representing "send nothing." Competes with targets.
- **Targets**: Each target token is a projection of `concat(global, source_knn, target)` — this means every target token "knows" the global state and which source planet is deciding.

> **Why this design?** By concatenating global + source context into every target token, we let the transformer focus on **comparing targets against each other** rather than figuring out the source context. The self-attention can then answer: "Given that I'm planet X with Y ships, which of these targets is most worth sending to?"

In [ ]:
d = cfg.model.embed_dim  # 128

with torch.no_grad():
    # Broadcast global + source_knn into each target position
    global_exp = global_emb.unsqueeze(1).expand(B, T, d)     # [1, 30, 128]
    source_exp = src_knn.unsqueeze(1).expand(B, T, d)        # [1, 30, 128]
    
    # Concatenate: each target sees global + source_knn + its own features
    token_input = torch.cat([global_exp, source_exp, tgt_emb], dim=-1)  # [1, 30, 384]
    
    # Project down to embed_dim
    target_tokens = policy.token_projection(token_input)  # [1, 30, 128]
    
    # Prepend CLS and NoOp (learnable parameters)
    cls_token = policy.cls_token.expand(B, 1, d)   # [1, 1, 128]
    noop_token = policy.noop_token.expand(B, 1, d)  # [1, 1, 128]
    
    tokens = torch.cat([cls_token, noop_token, target_tokens], dim=1)  # [1, 32, 128]

print("=== Stage 2: Token Construction ===")
print(f"\nToken projection: concat(global[{d}], source_knn[{d}], target[{d}]) = [{3*d}] -> [{d}]")
print(f"\nFinal token sequence: {list(tokens.shape)}")
print(f"  Token 0: CLS      (learnable, for value estimation)")
print(f"  Token 1: NoOp     (learnable, 'send nothing' option)")
print(f"  Token 2-{1+T}: Targets (projected from global+source+target features)")

# Show token norms (measure of "how activated" each token is)
norms = tokens[0].norm(dim=-1).numpy()
print(f"\nToken norms (L2):")
print(f"  CLS:  {norms[0]:.3f}")
print(f"  NoOp: {norms[1]:.3f}")
print(f"  Target norms (first 6): {norms[2:8].round(3)}")
print(f"  Padded targets (should be ~0 or small): {norms[-3:].round(3)}")

### 3.3 Stage 3: The Transformer — Attention Is All You Need (for targeting)

The 2-layer transformer with 4 attention heads processes the token sequence. This is where the magic happens: targets can attend to each other, learning patterns like:

- "If the best target is a high-production enemy planet, suppress interest in low-production neutrals"
- "If multiple targets are along the same heading, maybe only send to the nearest one"
- "If the NoOp token attends strongly to the source (via CLS), it means we should conserve ships"

We use **pre-LayerNorm** (norm before attention/FFN, not after), which is more stable for training.

In [ ]:
with torch.no_grad():
    # Prepare attention mask
    key_padding_mask = ~mask_t  # True = masked out (MHA convention)
    
    # TransformerBlock uses batch_first=True, so shape stays [B, seq_len, dim]
    x = tokens  # [1, 32, 128]
    
    print("=== Stage 3: Transformer Processing ===")
    print(f"Input shape: {list(x.shape)} (batch={x.shape[0]}, seq_len={x.shape[1]}, dim={x.shape[2]})")
    print(f"Attention mask: {key_padding_mask.shape} ({key_padding_mask.sum().item()} positions masked)")
    
    # Process through transformer blocks
    for i, block in enumerate(policy.transformer_blocks):
        x_before = x.clone()
        x = block(x, key_padding_mask=key_padding_mask)
        
        # Measure how much each layer changes the representations
        delta = (x - x_before).norm() / x_before.norm()
        print(f"\n  Layer {i}: relative change = {delta:.4f}")
        print(f"    Output range: [{x.min():.3f}, {x.max():.3f}]")
        print(f"    Output std:   {x.std():.3f}")
    
    # Final layer norm
    x = policy.final_ln(x)  # [1, 32, 128]
    
    print(f"\nAfter final LayerNorm: range=[{x.min():.3f}, {x.max():.3f}], std={x.std():.3f}")

print(f"\nTransformer output: {list(x.shape)}")
print(f"  x[:, 0, :] = CLS representation (for value head)")
print(f"  x[:, 1, :] = NoOp representation")
print(f"  x[:, 2:, :] = Target representations")


### 3.4 Visualizing Attention Patterns

Let's peek inside the transformer's attention heads to see what relationships it learns. Even with random weights, the mask structure is informative.

In [ ]:
# Extract attention weights from the first layer
with torch.no_grad():
    # Re-run through first layer's self-attention to get weights
    first_block = policy.transformer_blocks[0]
    
    # Get the input to the first layer
    x_in = tokens  # [1, 32, 128] (batch_first)
    
    # Pre-norm
    x_normed = first_block.ln1(x_in)
    
    # Self-attention with weights
    attn_out, attn_weights = first_block.attn(
        x_normed, x_normed, x_normed,
        key_padding_mask=~mask_t,
        need_weights=True,
        average_attn_weights=False  # get per-head weights
    )

# attn_weights shape: [batch, n_heads, seq_len, seq_len]
aw = attn_weights[0].numpy()  # [4, 32, 32]
n_valid_tokens = int(mask_t[0].sum().item())

print("=== Attention Patterns (Layer 0, all heads) ===")
print(f"Shape: {aw.shape} (heads, queries, keys)")
print(f"Valid tokens: {n_valid_tokens}\n")

token_labels = ['CLS', 'NoOp'] + [f'T{i}' for i in range(min(n_valid_tokens-2, 6))]
n_show = len(token_labels)

for head in range(min(4, cfg.model.n_heads)):
    print(f"  Head {head}:")
    print(f"    {'':>6}", end='')
    for label in token_labels:
        print(f"{label:>6}", end='')
    print()
    
    for q in range(n_show):
        print(f"    {token_labels[q]:>5} ", end='')
        for k in range(n_show):
            val = aw[head, q, k]
            if val > 0.15:
                print(f"  {val:.2f}", end='')
            else:
                print(f"     .", end='')
        print()
    print()


### 3.5 Stage 4: Output Heads — From Representations to Decisions

Three output heads decode the transformer's representations:

1. **Target Head**: Linear projection `[128] -> [1]` on tokens 1..T+1, producing logits for NoOp + each target. Invalid targets get `-inf`.
2. **Fraction Head**: Linear projection `[128] -> [5]` on target tokens only, producing logits over 5 fraction bins `[0.2, 0.4, 0.6, 0.8, 1.0]`.
3. **Value Head**: MLP `[128] -> [128] -> [1]` on the CLS token, estimating expected return.

> **Factored actions**: Target and fraction are separate distributions. This is crucial — it means the network can learn "always send to the best target" independently from "how many ships to commit." A joint distribution over T*5 actions would be much harder to learn.

In [ ]:
with torch.no_grad():
    # Run the full forward pass through the policy
    outputs = policy(
        global_t, src_scalars_t, src_pos_t,
        knn_scalars_t, knn_pos_t,
        tgt_scalars_t, tgt_pos_t,
        mask_t
    )

print("=== Stage 4: Output Heads ===")
print(f"\nTarget logits shape: {list(outputs.target_logits.shape)} (NoOp + {T} targets)")
print(f"Fraction logits shape: {list(outputs.fraction_logits.shape)} ({T} targets x 5 fractions)")
print(f"Value estimate: {outputs.value.item():.4f}")

# Show target logits with masking
tl = outputs.target_logits[0].numpy()
n_valid_targets = n_valid_tokens - 2  # subtract CLS and NoOp from mask

print(f"\n--- Target Selection Logits ---")
print(f"{'Index':>6} {'Logit':>8} {'Prob':>8}  Description")
print("-" * 60)

# Compute softmax over valid targets only
valid_logits = tl[:1 + n_valid_targets]  # NoOp + valid targets
probs = np.exp(valid_logits - valid_logits.max())
probs = probs / probs.sum()

print(f"{'0':>6} {tl[0]:>8.3f} {probs[0]:>8.3f}  NoOp (send nothing)")
for t in range(min(n_valid_targets, 8)):
    pid = decision.target_planet_ids[t]
    planet = state.planets_by_id[pid]
    owner_str = {-1: 'NEU', 0: 'OWN', 1: 'ENE'}[planet.owner]
    print(f"{t+1:>6} {tl[t+1]:>8.3f} {probs[t+1]:>8.3f}  "
          f"Planet {pid} [{owner_str}] ships={planet.ships} prod={planet.production}")

print(f"\n(Remaining {n_valid_targets - 8} targets omitted)" if n_valid_targets > 8 else "")
print(f"Invalid targets: masked to -inf (prob=0)")

In [ ]:
# Show fraction logits for the top target
top_target = int(np.argmax(valid_logits))

print(f"--- Fraction Logits (for top target: index {top_target}) ---")
if top_target == 0:
    print("Top action is NoOp -- fraction is irrelevant (zeroed out)")
else:
    frac_logits = outputs.fraction_logits[0, top_target - 1].numpy()  # [5]
    frac_probs = np.exp(frac_logits - frac_logits.max())
    frac_probs = frac_probs / frac_probs.sum()
    
    fractions = cfg.env.ship_fractions
    src_ships = src.ships
    
    print(f"{'Bin':>4} {'Fraction':>9} {'Ships':>6} {'Logit':>8} {'Prob':>8}  Visual")
    print("-" * 60)
    for i, (frac, logit, prob) in enumerate(zip(fractions, frac_logits, frac_probs)):
        ships = int(src_ships * frac)
        bar = '#' * int(prob * 40)
        print(f"{i:>4} {frac:>9.0%} {ships:>6} {logit:>8.3f} {prob:>8.3f}  {bar}")

print(f"\nValue estimate (from CLS token): {outputs.value.item():.4f}")
print(f"  (Untrained network, so this is essentially random noise)")

### 3.6 Sampling Actions — From Distributions to Moves

Now let's see how we go from logits to actual game moves. The `sample_actions` function creates two Categorical distributions and samples from them.

**The factored action decomposition**:
```
log_prob(action) = log_prob(target) + log_prob(fraction | target)
entropy(action) = entropy(target) + entropy(fraction)
```

This is valid because we treat target and fraction as independent (conditional independence given the transformer output).

In [ ]:
with torch.no_grad():
    # Stochastic sampling (for training)
    sampled = sample_actions(outputs, deterministic=False)
    
    # Deterministic (argmax) for evaluation
    sampled_det = sample_actions(outputs, deterministic=True)

print("=== Action Sampling ===")
print(f"\nStochastic sample (for training):")
print(f"  target_index: {sampled.target_index.item()} "
      f"({'NoOp' if sampled.target_index.item() == 0 else f'Planet {decision.target_planet_ids[sampled.target_index.item()-1]}'})")
print(f"  fraction_bin: {sampled.fraction_bin.item()} "
      f"({cfg.env.ship_fractions[sampled.fraction_bin.item()]:.0%} of ships)")
print(f"  log_prob: {sampled.log_prob.item():.4f} "
      f"(prob = {math.exp(sampled.log_prob.item()):.4f})")
print(f"  entropy: {sampled.entropy.item():.4f}")

print(f"\nDeterministic (argmax, for evaluation):")
print(f"  target_index: {sampled_det.target_index.item()} "
      f"({'NoOp' if sampled_det.target_index.item() == 0 else f'Planet {decision.target_planet_ids[sampled_det.target_index.item()-1]}'})")
print(f"  fraction_bin: {sampled_det.fraction_bin.item()} "
      f"({cfg.env.ship_fractions[sampled_det.fraction_bin.item()]:.0%} of ships)")

# Convert to game move
tgt_idx = sampled.target_index.item()
if tgt_idx == 0:
    print(f"\nGame move: [] (no action)")
else:
    target_offset = tgt_idx - 1
    frac = cfg.env.ship_fractions[sampled.fraction_bin.item()]
    ships = int(src.ships * frac)
    angle = decision.target_angles[target_offset]
    print(f"\nGame move: [{src.id}, {angle:.4f}, {ships}]")
    print(f"  = Send {ships} ships from Planet {src.id} at angle {math.degrees(angle):.1f} deg")

### 3.7 Full Forward Pass — End-to-End Summary

Let's trace the complete data flow one more time, now showing tensor shapes at every step:

In [ ]:
print("=" * 70)
print("COMPLETE FORWARD PASS: Observation -> Action")
print("=" * 70)

steps = [
    ("Raw observation", "Kaggle env", "planets, fleets, step, ..."),
    ("parse_observation()", "-> GameState", f"{len(state.planets)} planets, {len(state.fleets)} fleets"),
    ("compute_fleet_transit()", "-> FleetTransitState", f"{len(transit.transit)} planet transit entries"),
    ("encode_source_decision()", "-> SourceDecision", 
     f"global[{GLOBAL_DIM}] + src[2+{SOURCE_SCALAR_DIM}] + knn[{K}x(2+{KNN_SCALAR_DIM})] + tgt[{T}x(2+{TARGET_SCALAR_DIM})] + mask[{T+2}]"),
    ("pos_encoder(positions)", f"[2] -> [{d}]", "Shared spatial embedding"),
    ("global_encoder(global)", f"[{GLOBAL_DIM}] -> [{d}]", "Game state context"),
    ("source_encoder(pos+scalars)", f"[{d}+{SOURCE_SCALAR_DIM}] -> [{d}]", "Source planet embedding"),
    ("knn_encoder + mean_pool", f"[{K},{d}+{KNN_SCALAR_DIM}] -> [{d}]", "Neighborhood context"),
    ("source_knn_combiner", f"[{2*d}] -> [{d}]", "Combined source context"),
    ("target_encoder(pos+scalars)", f"[{T},{d}+{TARGET_SCALAR_DIM}] -> [{T},{d}]", "Target embeddings"),
    ("token_projection", f"[{T},{3*d}] -> [{T},{d}]", "Concat global+source+target, project"),
    ("prepend CLS + NoOp", f"-> [{T+2},{d}]", "Full token sequence"),
    ("transformer (2 layers)", f"[{T+2},{d}] -> [{T+2},{d}]", "Self-attention + FFN"),
    ("target_head", f"[{T+1},{d}] -> [{T+1}]", "Selection logits (NoOp + targets)"),
    ("fraction_head", f"[{T},{d}] -> [{T},5]", "Fraction logits per target"),
    ("value_head (from CLS)", f"[{d}] -> [1]", "State value estimate"),
    ("sample_actions()", "-> SampledAction", "target_index, fraction_bin, log_prob, entropy"),
]

for i, (name, shape, desc) in enumerate(steps):
    prefix = '  ' if i > 0 else ''
    arrow = '-> ' if i > 0 else '   '
    print(f"{prefix}{arrow}{name}")
    print(f"      {shape}  |  {desc}")

---

## Part 4: The Loss Functions — How the Agent Learns

We have three learning signals, used at different phases:

1. **Behavioral Cloning (BC) loss** — Cross-entropy against expert demonstrations
2. **PPO policy loss** — Clipped surrogate objective from RL
3. **Value loss** — MSE between predicted and actual returns

During DAgger training, the total loss is:
```
loss = policy_loss + 0.5 * value_loss - 0.01 * entropy + beta * bc_loss
```
where `beta` decays linearly from 0.5 to 0 over the first 1000 updates.

> **Why three losses?** Each serves a different purpose:
> - BC loss says "play like the expert" (fast but limited by expert quality)
> - PPO loss says "improve based on trial-and-error" (slow but unbounded)
> - Value loss trains the critic, which PPO needs for advantage estimation
> - Entropy bonus prevents premature convergence to a single strategy

In [ ]:
print("=== Loss Function Anatomy ===")
print()

# Let's compute each loss component manually to understand them

# --- 1. BC Loss (Behavioral Cloning) ---
# Suppose the expert chose target_index=2 (first real target) and fraction_bin=3 (80%)
expert_target = torch.tensor([2], dtype=torch.long)
expert_frac = torch.tensor([3], dtype=torch.long)

# Target cross-entropy
safe_logits = outputs.target_logits.clamp(min=-1e4)  # avoid -inf issues
target_ce = F.cross_entropy(safe_logits, expert_target)

# Fraction cross-entropy (only for non-NoOp)
frac_idx = (expert_target - 1).clamp(min=0)  # offset for fraction indexing
frac_logits_sel = outputs.fraction_logits[0, frac_idx[0]].unsqueeze(0)  # [1, 5]
frac_ce = F.cross_entropy(frac_logits_sel, expert_frac)

bc_loss = target_ce + frac_ce

print(f"1. BC Loss (imitation):")
print(f"   target_ce = {target_ce.item():.4f}  (expert chose target {expert_target.item()})")
print(f"   frac_ce   = {frac_ce.item():.4f}  (expert chose {cfg.env.ship_fractions[expert_frac.item()]:.0%})")
print(f"   bc_loss   = {bc_loss.item():.4f}")
print(f"   (For reference, random policy CE ~ -ln(1/{n_valid_tokens-1}) = {-math.log(1/(n_valid_tokens-1)):.2f})")

In [ ]:
# --- 2. PPO Policy Loss (Clipped Surrogate) ---

# Simulate: old policy had log_prob=-2.5, now we have new log_prob
old_log_prob = torch.tensor([-2.5])
new_log_prob = sampled.log_prob

# Advantage: how much better was this action than average?
advantage = torch.tensor([0.8])  # positive = better than expected

# Probability ratio
ratio = (new_log_prob - old_log_prob).exp()

# Clipped surrogate
clip_coef = cfg.ppo.clip_coef  # 0.2
surr1 = ratio * advantage
surr2 = torch.clamp(ratio, 1 - clip_coef, 1 + clip_coef) * advantage
policy_loss = -torch.min(surr1, surr2)  # Negative because we maximize

print(f"\n2. PPO Policy Loss (Clipped Surrogate):")
print(f"   old_log_prob = {old_log_prob.item():.4f}")
print(f"   new_log_prob = {new_log_prob.item():.4f}")
print(f"   ratio = exp(new - old) = {ratio.item():.4f}")
print(f"   advantage = {advantage.item():.4f}")
print(f"   unclipped = ratio * advantage = {surr1.item():.4f}")
print(f"   clipped   = clamp(ratio, {1-clip_coef}, {1+clip_coef}) * advantage = {surr2.item():.4f}")
print(f"   policy_loss = -min(unclipped, clipped) = {policy_loss.item():.4f}")
print(f"")
print(f"   Clip range: [{1-clip_coef}, {1+clip_coef}]")
print(f"   {'CLIPPED' if ratio.item() < 1-clip_coef or ratio.item() > 1+clip_coef else 'NOT CLIPPED'} "
      f"(ratio {'outside' if ratio.item() < 1-clip_coef or ratio.item() > 1+clip_coef else 'inside'} [{1-clip_coef:.1f}, {1+clip_coef:.1f}])")

In [ ]:
# --- 3. Value Loss ---
predicted_value = outputs.value  # from CLS token
actual_return = torch.tensor([0.3])  # discounted sum of future rewards

value_loss = 0.5 * (actual_return - predicted_value).pow(2)

print(f"3. Value Loss (MSE):")
print(f"   predicted V(s) = {predicted_value.item():.4f}")
print(f"   actual return  = {actual_return.item():.4f}")
print(f"   value_loss = 0.5 * ({actual_return.item():.4f} - {predicted_value.item():.4f})^2 = {value_loss.item():.6f}")

# --- 4. Entropy Bonus ---
entropy = sampled.entropy
print(f"\n4. Entropy Bonus:")
print(f"   entropy = {entropy.item():.4f}")
print(f"   max possible entropy (uniform over {n_valid_tokens-1} targets) = {math.log(n_valid_tokens-1):.4f}")
print(f"   entropy ratio = {entropy.item() / math.log(n_valid_tokens-1):.2%}")

# --- Total Loss ---
vf_coef = cfg.ppo.vf_coef  # 0.5
ent_coef = cfg.ppo.ent_coef  # 0.01
imitation_coef = 0.3  # mid-decay example

total_loss = (policy_loss.item() 
              + vf_coef * value_loss.item() 
              - ent_coef * entropy.item()
              + imitation_coef * bc_loss.item())

print(f"\n{'='*60}")
print(f"TOTAL LOSS = policy + {vf_coef}*value - {ent_coef}*entropy + {imitation_coef}*bc")
print(f"           = {policy_loss.item():.4f} + {vf_coef*value_loss.item():.4f} "
      f"- {ent_coef*entropy.item():.4f} + {imitation_coef*bc_loss.item():.4f}")
print(f"           = {total_loss:.4f}")

---

## Part 5: The DAgger Pipeline — Three Phases of Learning

DAgger (Dataset Aggregation) is our secret weapon for bootstrapping the RL agent. Instead of learning from scratch (which takes millions of steps), we first teach the agent to imitate an expert (the hybrid rule-based agent), then gradually transition to self-improvement through PPO.

```
Phase 1: Collect Demonstrations     Phase 2: BC Pretrain         Phase 3: PPO + Decaying Imitation
                                                                  
  Hybrid agent plays 50 games        30 epochs of supervised      PPO training with:
  against random opponent            cross-entropy learning       loss = PPO + beta * BC
  Records (state, action) pairs      on expert demonstrations     beta decays 0.5 -> 0 over 1000 updates
  Maps to our action space           ~493K params updated         Then pure PPO for remaining 2000 updates
```

> **Why not just do BC?** Behavioral cloning has a fundamental problem: **compounding errors**. The agent never sees states that result from its own mistakes, so when it makes a small error, it enters unfamiliar territory and makes bigger errors. DAgger addresses this by using BC as a starting point, then letting PPO correct mistakes through actual gameplay.

> **Why not just do PPO?** Pure PPO from scratch in a complex strategy game would take an enormous number of games. The agent would spend most of its time making random moves before accidentally discovering that capturing planets is good. BC gives it a massive head start.

In [ ]:
# Visualize the imitation coefficient schedule

coef_start = cfg.imitation.coef_start  # 0.5
coef_decay = cfg.imitation.coef_decay_updates  # 1000
total_updates = cfg.ppo.total_updates  # 3000

updates = list(range(0, total_updates + 1, 50))
coefs = [coef_start * max(0.0, 1.0 - u / coef_decay) for u in updates]

print("=== Imitation Coefficient Schedule ===")
print(f"Starts at {coef_start}, decays to 0 over {coef_decay} updates, then pure PPO for {total_updates-coef_decay} more\n")

# ASCII chart
width = 60
for i in range(0, len(updates), max(1, len(updates)//20)):
    u = updates[i]
    c = coefs[i]
    bar_len = int(c / coef_start * width)
    bar = '#' * bar_len + '.' * (width - bar_len)
    phase = "BC+PPO" if c > 0 else "Pure PPO"
    print(f"  update {u:>5} | beta={c:.3f} |{bar}| {phase}")

print(f"\nPhase boundaries:")
print(f"  Update 0: BC pretrain (30 epochs, lr=0.001)")
print(f"  Updates 1-{coef_decay}: PPO + imitation (beta: {coef_start} -> 0, lr={cfg.ppo.lr})")
print(f"  Updates {coef_decay+1}-{total_updates}: Pure PPO (lr={cfg.ppo.lr})")

### 5.1 Phase 1: Demonstration Collection

The hybrid agent (a sophisticated rule-based agent with mission planning and timeline management) plays 50 games against random opponents. For each step, we record what the hybrid chose and map it to our action space.

The mapping has a tolerance of 30 degrees — if the expert's aim angle is within 30 degrees of one of our target angles, we record it as that target. Otherwise, we record it as NoOp.

This is a lossy mapping — the hybrid agent's continuous actions don't perfectly align with our discrete action space. This is a known limitation.

In [ ]:
# Demonstrate the action space mapping

print("=== Action Space Mapping ===")
print(f"Expert (hybrid agent) outputs: [from_planet_id, angle_radians, num_ships]")
print(f"Our action space: (target_index, fraction_bin)")
print(f"Fraction bins: {cfg.env.ship_fractions}")
print()

# Show how an expert move maps to our space
expert_angle = 1.2  # radians
expert_ships = 30
src_ships = 45

print(f"Example: Expert sends {expert_ships} ships at angle {math.degrees(expert_angle):.1f} deg")
print(f"         from planet with {src_ships} ships")
print(f"")
print(f"Step 1: Find closest target by angle (tolerance 30 deg)")

for i, tgt_angle in enumerate(decision.target_angles[:6]):
    diff = abs(expert_angle - tgt_angle)
    diff = min(diff, 2 * math.pi - diff)  # wrap around
    match = ' <-- MATCH' if math.degrees(diff) < 30 else ''
    print(f"  Target {i} angle={math.degrees(tgt_angle):.1f} deg, diff={math.degrees(diff):.1f} deg{match}")

print(f"\nStep 2: Map ship fraction")
frac = expert_ships / src_ships
print(f"  Fraction = {expert_ships}/{src_ships} = {frac:.2f}")
for i, f in enumerate(cfg.env.ship_fractions):
    diff = abs(f - frac)
    best = ' <-- NEAREST' if i == min(range(5), key=lambda j: abs(cfg.env.ship_fractions[j] - frac)) else ''
    print(f"  Bin {i} ({f:.0%}): diff = {diff:.2f}{best}")

### 5.2 The GAE Advantage Computation

PPO needs **advantages** — how much better/worse was this action compared to the average? We use **Generalized Advantage Estimation (GAE)** which balances bias and variance through the lambda parameter.

A critical subtlety: **multiple planet decisions share the same reward**. When 3 planets each make a decision on the same turn, they all receive the same step reward. This means all three decisions are equally credited/blamed.

> **This is a significant weakness**: the credit assignment problem. If planet A makes a brilliant move and planet B makes a terrible move on the same turn, they both get the same advantage signal. The network must learn to distinguish good from bad decisions despite this noise.

In [ ]:
# Demonstrate GAE computation

gamma = cfg.ppo.gamma  # 0.99
lam = cfg.ppo.gae_lambda  # 0.95

print("=== GAE (Generalized Advantage Estimation) ===")
print(f"gamma={gamma}, lambda={lam}")
print()

# Simulate a 5-step episode with 2 planets per step
rewards = [0.0, 0.0, 0.005, 0.0, 1.0]  # dense + terminal win
values = [0.3, 0.35, 0.4, 0.5, 0.6]     # value estimates (getting more optimistic)
bootstrap_value = 0.0  # terminal state
n_planets_per_step = [2, 3, 2, 1, 2]     # variable number of decisions

print("Step-by-step return computation (backward pass):")
print("-" * 70)

future_return = bootstrap_value
returns = []
advantages = []

for t in range(len(rewards) - 1, -1, -1):
    r = rewards[t]
    done = 1.0 if t == len(rewards) - 1 else 0.0
    future_return = r + gamma * future_return * (1.0 - done)
    adv = future_return - values[t]
    
    n_planets = n_planets_per_step[t]
    returns.insert(0, future_return)
    advantages.insert(0, adv)
    
    print(f"  Step {t}: reward={r:.3f}, V(s)={values[t]:.3f}, "
          f"return={future_return:.4f}, advantage={adv:+.4f}, "
          f"assigned to {n_planets} planet decisions")

print(f"\nKey observation: Step 2 had reward=0.005 and 2 planet decisions.")
print(f"Both planets get advantage={advantages[2]:+.4f}, even though one might")
print(f"have been responsible for the good outcome and the other wasn't.")
print(f"This is the credit assignment problem in multi-agent-like settings.")

---

## Part 6: The Reward Model — Strengths and Weaknesses

### Current Reward Structure

The DAgger config uses **dense reward** (sparse reward is also available):

```
reward_per_step = delta_ships * 0.001 + delta_production * 0.005
reward_terminal = +1 (win) or -1 (loss) or 0 (tie)
```

This is simple but has deep implications:

In [ ]:
print("=== Reward Model Analysis ===")
print()

# What does the dense reward signal look like?
scenarios = [
    ("Capture a prod-3 planet (15 ships)", 15, 3),
    ("Capture a prod-1 planet (5 ships)", 5, 1),
    ("Lose 20 ships attacking (failed)", -20, 0),
    ("Gain 5 ships from production", 5, 0),
    ("Lose a prod-2 planet (-10 ships)", -10, -2),
    ("Send 30 ships (in transit)", -30, 0),
]

ship_coef = cfg.reward.dense_ship_coef  # 0.001
prod_coef = cfg.reward.dense_prod_coef  # 0.005

print(f"Coefficients: ship_coef={ship_coef}, prod_coef={prod_coef}")
print(f"{'Scenario':<45} {'delta_ships':>12} {'delta_prod':>11} {'reward':>8}")
print("-" * 80)

for desc, ds, dp in scenarios:
    r = ds * ship_coef + dp * prod_coef
    print(f"{desc:<45} {ds:>+12} {dp:>+11} {r:>+8.4f}")

print(f"\nFor comparison:")
print(f"  Terminal reward: +1.0 (win) vs -1.0 (loss)")
print(f"  Capturing a prod-5 planet: {50*ship_coef + 5*prod_coef:+.4f} per turn")
print(f"  Over 500 turns of production: {500*5*ship_coef:+.4f} cumulative ship reward")

In [ ]:
print("\n=== Reward Model: Strengths ===")
print("""
1. DENSE SIGNAL: Unlike sparse (terminal-only) reward, the agent gets feedback
   every turn. This dramatically speeds up learning — the agent doesn't need to
   randomly stumble into a win to start learning.

2. PRODUCTION WEIGHTED HIGHER: prod_coef (0.005) > ship_coef (0.001), which
   correctly biases toward capturing high-production planets over hoarding ships.
   A prod-5 planet gives 5 * 0.005 = 0.025/turn vs 5 * 0.001 = 0.005 for ships.

3. TERMINAL DOMINANCE: The +/-1 terminal reward dwarfs the dense rewards,
   so the agent ultimately optimizes for winning, not just accumulating ships.
""")

print("=== Reward Model: Weaknesses ===")
print("""
1. SHIPS IN TRANSIT ARE INVISIBLE: When you launch a fleet, you lose ships
   from your planet (negative reward) but don't gain them until the fleet
   arrives. This creates a NEGATIVE REWARD for attacking, which the agent
   must learn to overcome through delayed gratification. This is hard.

2. NO STRATEGIC SHAPING: The reward doesn't distinguish between:
   - Capturing a planet near the enemy (strategically valuable)
   - Capturing an isolated planet (less valuable)
   - Defending a key chokepoint vs. a useless outpost

3. CREDIT ASSIGNMENT: All planet decisions in a single turn share the
   same reward. If planet A makes a great move and planet B wastes ships,
   they get the same advantage signal.

4. TEMPORAL CREDIT: A fleet launched at step 100 might capture a planet
   at step 115. The reward appears 15 steps later with no direct link
   to the launch decision. GAE helps but gamma^15 * reward is very small.
""")

---

## Part 7: Strengths, Weaknesses, and the Path Forward

Now that we've seen every component, let's step back and assess the architecture honestly.

In [ ]:
print("""  
╔══════════════════════════════════════════════════════════════════════╗
║                    ARCHITECTURE STRENGTHS                          ║
╚══════════════════════════════════════════════════════════════════════╝

1. STRONG INDUCTIVE BIASES
   - Per-planet sequential decisions mirror human reasoning
   - Transit state tracking prevents redundant fleet launches
   - Sun avoidance baked into the mask (impossible to send through sun)
   - Targets sorted by distance (nearby = more relevant)

2. EFFICIENT ARCHITECTURE
   - ~493K params is tiny by modern standards
   - 2-layer transformer is fast (~1ms inference)
   - Factored actions reduce action space from T*5 to T + 5
   - Shared position encoder reduces parameters

3. DAGGER BOOTSTRAP
   - Doesn't start from scratch — leverages expert knowledge
   - Gradual transition prevents catastrophic forgetting
   - Distilled opponent provides consistent training signal

4. RICH FEATURES
   - Fleet transit predictions (who's going where)
   - KNN neighborhood context (am I safe?)
   - Global game state (am I winning?)
   - Orbit prediction for moving planets

╔══════════════════════════════════════════════════════════════════════╗
║                    ARCHITECTURE WEAKNESSES                         ║
╚══════════════════════════════════════════════════════════════════════╝

1. CREDIT ASSIGNMENT
   - All planets in a turn share the same reward signal
   - No per-planet advantage estimation
   - Fleet results arrive many steps after launch decisions

2. SEQUENTIAL DECISIONS ARE ORDER-DEPENDENT
   - Processing planets by ship count (desc) is an arbitrary choice
   - The first planet to decide gets the "cleanest" transit state
   - Later planets are constrained by earlier decisions
   - No mechanism to "undo" or revise earlier decisions

3. SINGLE-STEP HORIZON
   - Each forward pass decides for ONE turn only
   - No explicit multi-turn planning ("send fleet A now, fleet B in 5 turns")
   - The value function must implicitly encode future plans

4. LIMITED ACTION SPACE
   - Only 5 fraction bins (20/40/60/80/100%)
   - Can only target the 30 nearest planets
   - Only one fleet per source planet per turn (no split sends)

5. EXPERT BOTTLENECK
   - BC quality is limited by the hybrid agent's strength
   - Action space mapping is lossy (30-degree angular tolerance)
   - Expert doesn't model the RL agent's own capabilities
""")

### 7.1 Paths to Improvement

Here are concrete ideas ranked by expected impact and implementation effort:

In [ ]:
improvements = [
    ("HIGH IMPACT / LOW EFFORT", [
        ("Self-play training",
         "Train against copies of yourself. Already implemented — just set opponent='self'."
         " Prevents overfitting to competitive agent's weaknesses.",
         "configs: opponent: self"),
        ("Reward shaping: fleet launch credit",
         "Give immediate positive reward when launching a fleet that will arrive"
         " at an enemy/neutral planet. Discounted by travel time. Addresses the"
         " 'negative reward for attacking' problem.",
         "Modify src/env.py: _compute_reward()"),
        ("Larger model",
         "embed_dim=256, n_layers=4, ff_dim=512 (~2M params). With GPU training"
         " this is basically free. More capacity for strategic reasoning.",
         "configs: model.embed_dim: 256, model.n_layers: 4"),
    ]),
    ("HIGH IMPACT / MEDIUM EFFORT", [
        ("Per-planet value heads (multi-agent credit assignment)",
         "Instead of one value from CLS, have each target token predict its own"
         " advantage. Each planet's decision gets its own baseline, improving"
         " credit assignment. Similar to MAPPO's centralized-critic approach.",
         "Modify src/policy.py + src/ppo.py"),
        ("Population-based training (league)",
         "Maintain a pool of 5-10 past checkpoints. Sample opponents from the"
         " pool with probability proportional to recency. Prevents forgetting"
         " old strategies while learning new ones.",
         "Modify src/train.py + src/opponents.py"),
        ("Comet tracking features",
         "Comets spawn at predictable intervals with known paths. Add comet"
         " position predictions and ownership to target features. Free production!",
         "Modify src/features.py"),
    ]),
    ("HIGH IMPACT / HIGH EFFORT", [
        ("Offline RL (Decision Transformer or CQL)",
         "Collect a large dataset of games (expert + self-play + random). Train"
         " a Decision Transformer that conditions on desired return. No need for"
         " online rollouts — much more sample efficient. The current architecture"
         " is already transformer-based, making this a natural extension.",
         "New: src/offline_rl.py"),
        ("Meta-learning (MAML or RL^2)",
         "Learn to adapt to opponent style within a game. Use the first ~50 turns"
         " to identify opponent type (aggressive, defensive, random), then adapt"
         " strategy. Could be implemented as a recurrent state that persists"
         " across turns.",
         "Modify src/policy.py: add LSTM/GRU state"),
        ("Attention-based multi-planet coordination",
         "Instead of sequential per-planet decisions, use a second transformer"
         " that attends over all owned planets simultaneously. Each planet-token"
         " can attend to other planet-tokens, enabling coordinated attacks.",
         "Major redesign of src/policy.py + src/train.py"),
    ]),
]

for category, items in improvements:
    print(f"\n{'='*70}")
    print(f"  {category}")
    print(f"{'='*70}")
    for name, desc, where in items:
        print(f"\n  >> {name}")
        print(f"     {desc}")
        print(f"     Where: {where}")

### 7.2 Offline RL & Meta-Learning: The Frontier

Two approaches that could fundamentally change the agent's capabilities:

**Offline RL (Decision Transformer)**

The key insight: we already have a transformer architecture. A Decision Transformer would condition on **desired return** as an input, allowing us to control how aggressively the agent plays at test time.

```
Standard:    state -> policy -> action
DT:          (return_to_go, state) -> policy -> action
```

At inference, set `return_to_go = 1.0` ("I want to win"). The model learns from a dataset of both wins and losses, extracting the strategies that lead to high returns.

**Advantages**: No online rollouts needed. Can learn from any data source (replays, expert games, random games). Very sample efficient.

**Meta-Learning (RL^2)**

Add a recurrent component (GRU) that persists across turns within a game. The GRU hidden state becomes an "opponent model" — after observing enemy behavior for 50 turns, the agent implicitly adapts its strategy.

```
Turn 1:   state -> GRU(h_0) -> policy -> action, h_1
Turn 2:   state -> GRU(h_1) -> policy -> action, h_2
...
Turn 50:  state -> GRU(h_49) -> policy -> action, h_50  (now adapted!)
```

**Advantages**: Adapts to opponent style in real-time. No separate opponent classification needed.

---

## Part 8: Putting It All Together — A Complete Training Step

Let's simulate one complete training iteration to see how all the pieces fit together.

In [ ]:
# Simulate a mini rollout to show the full training loop
policy.train()
optimizer = torch.optim.Adam(policy.parameters(), lr=cfg.ppo.lr)

print("=== Simulated Training Step ===")
print()

# 1. Collect rollout data (we'll make 4 decisions)
n_decisions = len(my_planets)  # one decision per owned planet
all_features = []
all_actions = []
all_log_probs = []
all_values = []

# Reset transit for clean collection
transit = compute_fleet_transit(state)

print("Step 1: Collect rollout")
for i, src_planet in enumerate(my_planets[:n_decisions]):
    dec = encode_source_decision(src_planet, state, transit, cfg.env)
    
    # Tensorize
    feats = {
        'global': torch.tensor(dec.global_features, dtype=torch.float32).unsqueeze(0),
        'src_s': torch.tensor(dec.source_scalars, dtype=torch.float32).unsqueeze(0),
        'src_p': torch.tensor(dec.source_position, dtype=torch.float32).unsqueeze(0),
        'knn_s': torch.tensor(dec.knn_scalars, dtype=torch.float32).unsqueeze(0),
        'knn_p': torch.tensor(dec.knn_positions, dtype=torch.float32).unsqueeze(0),
        'tgt_s': torch.tensor(dec.target_scalars, dtype=torch.float32).unsqueeze(0),
        'tgt_p': torch.tensor(dec.target_positions, dtype=torch.float32).unsqueeze(0),
        'mask': torch.tensor(dec.target_mask, dtype=torch.bool).unsqueeze(0),
    }
    
    with torch.no_grad():
        out = policy(feats['global'], feats['src_s'], feats['src_p'],
                     feats['knn_s'], feats['knn_p'],
                     feats['tgt_s'], feats['tgt_p'], feats['mask'])
        act = sample_actions(out, deterministic=False)
    
    all_features.append(feats)
    all_actions.append(act)
    all_log_probs.append(act.log_prob.item())
    all_values.append(out.value.item())
    
    tgt_name = 'NoOp' if act.target_index.item() == 0 else f'Target {act.target_index.item()}'
    frac = cfg.env.ship_fractions[act.fraction_bin.item()]
    print(f"  Decision {i}: Planet {src.id} -> {tgt_name}, "
          f"fraction={frac:.0%}, log_prob={act.log_prob.item():.3f}, V={out.value.item():.3f}")

# Simulate rewards
rewards = [0.0] * n_decisions  # mostly zero with one small dense reward
returns = [0.3 + i * 0.02 for i in range(n_decisions)]  # computed from GAE (simulated)
advantages = [r - v for r, v in zip(returns, all_values)]

print(f"\n  Rewards: {rewards}")
print(f"  Returns (GAE): {[f'{r:.3f}' for r in returns]}")
print(f"  Advantages: {[f'{a:+.3f}' for a in advantages]}")


In [ ]:
# 2. PPO update on collected data
print("\nStep 2: PPO update")
print(f"  Epochs: {cfg.ppo.epochs}")
print(f"  Clip coefficient: {cfg.ppo.clip_coef}")

# Normalize advantages
adv_tensor = torch.tensor(advantages, dtype=torch.float32)
adv_norm = (adv_tensor - adv_tensor.mean()) / (adv_tensor.std() + 1e-8)
print(f"  Raw advantages: {[f'{a:+.3f}' for a in advantages]}")
print(f"  Normalized:     {[f'{a.item():+.3f}' for a in adv_norm]}")

returns_tensor = torch.tensor(returns, dtype=torch.float32)

# One epoch of PPO
total_policy_loss = 0
total_value_loss = 0
total_entropy = 0

for i in range(n_decisions):
    feats = all_features[i]
    old_act = all_actions[i]
    old_lp = all_log_probs[i]
    
    # Forward pass (with gradients this time)
    out = policy(feats['global'], feats['src_s'], feats['src_p'],
                 feats['knn_s'], feats['knn_p'],
                 feats['tgt_s'], feats['tgt_p'], feats['mask'])
    
    new_lp, ent = action_log_prob_and_entropy(
        out, old_act.target_index, old_act.fraction_bin
    )
    
    # PPO loss
    ratio = (new_lp - old_lp).exp()
    a = adv_norm[i]
    surr1 = ratio * a
    surr2 = torch.clamp(ratio, 1 - cfg.ppo.clip_coef, 1 + cfg.ppo.clip_coef) * a
    pol_loss = -torch.min(surr1, surr2)
    
    # Value loss
    val_loss = 0.5 * (returns_tensor[i] - out.value).pow(2)
    
    total_policy_loss += pol_loss.item()
    total_value_loss += val_loss.item()
    total_entropy += ent.item()

print(f"\n  Avg policy loss: {total_policy_loss/n_decisions:.4f}")
print(f"  Avg value loss:  {total_value_loss/n_decisions:.4f}")
print(f"  Avg entropy:     {total_entropy/n_decisions:.4f}")
print(f"\n  (In real training, this runs for {cfg.ppo.epochs} epochs")
print(f"   over {cfg.ppo.rollout_steps * cfg.ppo.num_envs} decisions per update)")

---

## Part 9: Key Takeaways

### What We Built

A **Transformer DAgger PPO agent** that:
1. Parses raw game observations into structured features with domain knowledge
2. Encodes those features into a token sequence (CLS + NoOp + Targets)
3. Processes tokens through a 2-layer transformer with masked attention
4. Outputs factored actions (target selection + ship fraction) and a state value
5. Learns via 3-phase DAgger: BC pretrain -> PPO + imitation decay -> Pure PPO

### The Big Ideas

1. **Feature engineering dominates**: The 80+ features per decision encode decades of game AI wisdom (transit prediction, sun avoidance, orbit extrapolation). Without these, the transformer would need orders of magnitude more data.

2. **Inductive biases are free performance**: Per-planet decisions, sorted targets, shared position encoders — each constrains the model but massively reduces the problem's complexity.

3. **DAgger bridges the expert-RL gap**: Pure BC is limited by the expert. Pure PPO is too slow. DAgger gets the best of both worlds: start with expert knowledge, then improve beyond it.

4. **The transformer's role is comparison**: Unlike NLP transformers that model sequences, our transformer compares targets against each other. Self-attention lets the "best" target suppress competitors.

5. **Credit assignment is the bottleneck**: The biggest weakness isn't the architecture — it's that multiple decisions share rewards. Solving this (per-planet critics, counterfactual baselines) would likely yield the biggest improvement.

### What Would Karpathy Say?

> *"The most important thing in deep learning is to look at your data. Print your tensors. Visualize your attention patterns. The bugs that matter are never syntax errors — they're silent numerical issues where everything runs but nothing learns."*

This notebook showed you every tensor, every dimension, every loss component. When your agent isn't learning, come back here and check: are the features sensible? Are the logits reasonable? Is the advantage positive for good actions? The answers are always in the numbers.

In [ ]:
# Final summary: model card
print("╔══════════════════════════════════════════════════════════════╗")
print("║              TRANSFORMER DAGGER PPO — MODEL CARD           ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  Parameters:        {count_params(policy):>10,}                         ║")
print(f"║  Embedding dim:     {cfg.model.embed_dim:>10}                         ║")
print(f"║  Transformer layers:{cfg.model.n_layers:>10}                         ║")
print(f"║  Attention heads:   {cfg.model.n_heads:>10}                         ║")
print(f"║  FF dim:            {cfg.model.ff_dim:>10}                         ║")
print(f"║  Max targets:       {cfg.env.max_targets:>10}                         ║")
print(f"║  Fraction bins:     {len(cfg.env.ship_fractions):>10}                         ║")
print(f"║  Action space:      {cfg.env.max_targets+1:>10} targets x 5 fracs     ║")
print(f"║  BC games:          {cfg.imitation.bc_games:>10}                         ║")
print(f"║  BC epochs:         {cfg.imitation.bc_epochs:>10}                         ║")
print(f"║  PPO updates:       {cfg.ppo.total_updates:>10}                         ║")
print(f"║  Learning rate:     {cfg.ppo.lr:>10}                         ║")
print(f"║  Imitation decay:   {cfg.imitation.coef_decay_updates:>10} updates                  ║")
print("╚══════════════════════════════════════════════════════════════╝")